# Tutorial 2: Building a Research Object Crate

In [Tutorial 1](01_dexpi_fair_intro.ipynb) we turned a DEXPI Proteus XML
file into a Python model and built a knowledge graph from it. The data is
already *Interoperable* and *Reusable*; but in order to contextualize it,
i. e. merge the structural data with operational data we need to **link**
and **package** it together - also addign other contextual data:
- **provenance data**: who produced it, with which equipment, under which experiment, and which
analyses were performed #TODO: refine
- **license data** #TODO: refine
thus also ruling besides interoperability, and reusability, every aspect of "accessibility" #TODO: refine

That bundle is what an **RO-Crate** (Research Object Crate) gives us.

## Learning goals

By the end of this notebook you will:

1. Understand what an **RO-Crate** is and why it matters for FAIR RDM,
2. Use [`rocrate`](https://github.com/ResearchObject/ro-crate-py) to
   programmatically build a crate,
3. Map **DEXPI equipment** (extracted with `pyDEXPI`) to RO-Crate
   contextual entities,
4. Attach **PAT sensor data** and a small **analysis** as crate datasets,
   linked back to the equipment they describe,
5. Produce a self-contained, shareable folder
   (`ro_crate_output/plant_demo/`) containing every artefact and a
   machine-readable `ro-crate-metadata.json` index


## What is an RO-Crate?

> An RO-Crate is **a folder** (or zip file) that follows a small
> convention: at the root sits a JSON-LD file called
> `ro-crate-metadata.json` that describes everything in the folder using
> [schema.org](https://schema.org/) vocabulary.

That single convention solves an enormous problem in research data
management: a stranger (or a future you) can pick up the folder years from
now, read the metadata file, and immediately know what each file is, who
produced it, and how it relates to everything else.

## 1 Target structure

We will build the following layout under `ro_crate_output/plant_demo/`:

```
plant_demo/
├── ro-crate-metadata.json        ← the JSON-LD index (auto-generated)
├── plant_setup/
│   └── C01V04-VER.EX01.xml       ← the original DEXPI file
├── data/
│   └── temperature_T4750.csv     ← PAT data, linked to equipment T4750
└── analysis/
    ├── temperature_profile.py    ← the analysis SCRIPT (re-runnable!)
    ├── temperature_profile.png   ← figure produced by the script
    └── temperature_profile.md    ← markdown report produced by the script
```

In addition, the metadata file will declare each tagged DEXPI equipment
(Pumps, Heat Exchangers, Tank) as a **contextual entity** of type
`IndividualProduct`, so the temperature data file can explicitly state
*"this measurement is about equipment **T4750**"*. The analysis script will
be tied to its inputs and outputs through a `CreateAction`, giving us a
complete, machine-readable provenance chain.


## 2 Setup


In [24]:
import shutil
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pydexpi.loaders import ProteusSerializer
from rocrate.rocrate import ROCrate

# Paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DEXPI_FILE   = PROJECT_ROOT / "data" / "dexpi"   / "C01V04-VER.EX01.xml"
PAT_FILE     = PROJECT_ROOT / "data" / "pat_data" / "sample_experiment.csv"
CRATE_DIR    = PROJECT_ROOT / "ro_crate_output"  / "plant_demo"

# Start from a clean slate so the notebook is fully reproducible
if CRATE_DIR.exists():
    shutil.rmtree(CRATE_DIR)
CRATE_DIR.mkdir(parents=True)

print(f"Crate target : {CRATE_DIR.relative_to(PROJECT_ROOT)}")
print(f"DEXPI file   : {DEXPI_FILE.relative_to(PROJECT_ROOT)} (exists={DEXPI_FILE.exists()})")
print(f"PAT data     : {PAT_FILE.relative_to(PROJECT_ROOT)} (exists={PAT_FILE.exists()})")


Crate target : ro_crate_output/plant_demo
DEXPI file   : data/dexpi/C01V04-VER.EX01.xml (exists=True)
PAT data     : data/pat_data/sample_experiment.csv (exists=True)


## 3 Re-load the DEXPI plant and extract the equipment list

We re-use exactly the same code we wrote in Tutorial 1. From `pyDEXPI` we
get a typed list of equipment items, each with a tag (`P4711`, `T4750`, …)
and a UUID. Those will become the **research objects** that describe our
plant setup inside the crate.


In [25]:
dexpi_model = ProteusSerializer().load(str(DEXPI_FILE.parent), DEXPI_FILE.name)

equipment = [
    {"tag": item.tagName, "type": type(item).__name__, "uuid": item.id}
    for item in dexpi_model.conceptualModel.taggedPlantItems
]
pd.DataFrame(equipment)


,tag,type,uuid
0,H1007,PlateHeatExchanger,801b1dc7-9898-4869-9db2-eadd4a994318
1,H1008,TubularHeatExchanger,d2f6e956-0ece-402e-be6f-37cfcae65f66
2,P4711,CentrifugalPump,5bf668a4-8ae3-40f6-a695-1a458646e2be
3,P4712,ReciprocatingPump,d6e369a8-4750-4dcc-b737-6f2578bf096a
4,T4750,Tank,aca8c415-abdb-4f43-b8e1-0be85054f207


### 3.1 Pick the equipment our experiment is about

For this tutorial we'll pretend our experiment instrumented the **tank
`T4750`** with a temperature sensor. We re-tag the bundled sample
time-series accordingly so that the IDs in the data file match the IDs in
the plant — a small but crucial detail for FAIR data: **identifiers must
be consistent across artefacts**.


In [26]:
TARGET_EQUIPMENT_TAG = "T4750"

raw = pd.read_csv(PAT_FILE, parse_dates=["timestamp"])
raw["equipment_id"] = TARGET_EQUIPMENT_TAG  # align IDs with the DEXPI plant
print(f"Rows: {len(raw)}, sensors: {raw['sensor_type'].unique().tolist()}")
raw.head()


Rows: 400, sensors: ['temperature', 'pH', 'pressure', 'flow']


,timestamp,equipment_id,sensor_type,value,unit
0,2024-01-01 10:00:00,T4750,temperature,36.810473,celsius
1,2024-01-01 10:05:00,T4750,temperature,37.382642,celsius
2,2024-01-01 10:10:00,T4750,temperature,36.267447,celsius
3,2024-01-01 10:15:00,T4750,temperature,37.063667,celsius
4,2024-01-01 10:20:00,T4750,temperature,37.425371,celsius


## 4 Stage the files into the crate folder

We need to *physically* place every artefact under `CRATE_DIR` before
registering it. RO-Crate is just a folder + a JSON-LD index, so the layout
on disk matters.


In [27]:
import subprocess, sys, textwrap

# --- 4.1 plant_setup/ : copy the DEXPI XML ----------------------------------
plant_setup_dir = CRATE_DIR / "plant_setup"
plant_setup_dir.mkdir(exist_ok=True)
dexpi_in_crate = plant_setup_dir / DEXPI_FILE.name
shutil.copy(DEXPI_FILE, dexpi_in_crate)

# --- 4.2 data/ : write the (re-tagged) PAT data ------------------------------
data_dir = CRATE_DIR / "data"
data_dir.mkdir(exist_ok=True)
temperature_csv = data_dir / f"temperature_{TARGET_EQUIPMENT_TAG}.csv"
raw[raw["sensor_type"] == "temperature"].to_csv(temperature_csv, index=False)

# --- 4.3 analysis/ : write the analysis SCRIPT ------------------------------
# We keep the *executable* analysis next to the data it consumes. Anyone with
# Python + pandas + matplotlib can re-run it on the CSV in the crate.
analysis_dir = CRATE_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)

analysis_script = analysis_dir / "temperature_profile.py"
analysis_script.write_text(textwrap.dedent('''\
    """Temperature profile analysis.

    Reads a CSV of temperature measurements (columns: timestamp, equipment_id,
    sensor_type, value, unit) and produces:
      * temperature_profile.png — line plot of value vs time
      * temperature_profile.md  — short markdown report with descriptive stats

    Usage
    -----
        python temperature_profile.py <input_csv> [<output_dir>]

    If <output_dir> is omitted, files are written next to the script.
    """
    from __future__ import annotations

    import sys
    from pathlib import Path

    import matplotlib.pyplot as plt
    import pandas as pd


    def run(csv_path: Path, out_dir: Path) -> tuple[Path, Path]:
        df = pd.read_csv(csv_path, parse_dates=["timestamp"]).sort_values("timestamp")
        equipment = df["equipment_id"].iloc[0]

        # --- plot --------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.plot(df["timestamp"], df["value"], color="#e74c3c", lw=1.2)
        ax.set_xlabel("Time")
        ax.set_ylabel("Temperature [\u00b0C]")
        ax.set_title(f"Temperature profile of equipment {equipment}")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        png_path = out_dir / "temperature_profile.png"
        fig.savefig(png_path, dpi=120)
        plt.close(fig)

        # --- markdown report ---------------------------------------------------
        s = df["value"].describe()[["mean", "std", "min", "max"]].round(2)
        md_path = out_dir / "temperature_profile.md"
        md_path.write_text(
            f"# Temperature profile analysis\\n\\n"
            f"Summary of the temperature time-series for equipment "
            f"**{equipment}**.\\n\\n"
            f"| Statistic | Value (\u00b0C) |\\n"
            f"|-----------|------------|\\n"
            f"| Mean      | {s['mean']} |\\n"
            f"| Std. dev. | {s['std']} |\\n"
            f"| Minimum   | {s['min']} |\\n"
            f"| Maximum   | {s['max']} |\\n\\n"
            f"Generated by `temperature_profile.py` from `{csv_path.name}`.\\n"
        )
        return png_path, md_path


    if __name__ == "__main__":
        if len(sys.argv) < 2:
            print(__doc__)
            sys.exit(1)
        csv = Path(sys.argv[1]).resolve()
        out = Path(sys.argv[2]).resolve() if len(sys.argv) > 2 else csv.parent
        out.mkdir(parents=True, exist_ok=True)
        png, md = run(csv, out)
        print(f"wrote {png}")
        print(f"wrote {md}")
'''))

# --- 4.4 Run the script so the crate ships pre-computed outputs -------------
# This is the *real* provenance link: the PNG and MD that end up in the crate
# are produced by exactly the script we just wrote.
result = subprocess.run(
    [sys.executable, str(analysis_script), str(temperature_csv), str(analysis_dir)],
    capture_output=True, text=True, check=True,
)
print(result.stdout)

profile_png = analysis_dir / "temperature_profile.png"
report_md   = analysis_dir / "temperature_profile.md"

print("Crate now contains:")
for p in sorted(CRATE_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(CRATE_DIR)}  ({p.stat().st_size/1024:.1f} KB)")


wrote /Users/niklas-maximilianepping/Projekte/PSE-KG/ro_crate_output/plant_demo/analysis/temperature_profile.png
wrote /Users/niklas-maximilianepping/Projekte/PSE-KG/ro_crate_output/plant_demo/analysis/temperature_profile.md

Crate now contains:
  analysis/temperature_profile.md  (0.3 KB)
  analysis/temperature_profile.png  (62.4 KB)
  analysis/temperature_profile.py  (2.2 KB)
  data/temperature_T4750.csv  (6.3 KB)
  plant_setup/C01V04-VER.EX01.xml  (435.3 KB)


## 5 Build the RO-Crate metadata

Now the fun part: we describe each artefact in machine-readable form using
[`rocrate`](https://github.com/ResearchObject/ro-crate-py). Two kinds of
entities are added:

- **Data entities** — actual files/folders inside the crate (XML, CSV, PNG, MD).
- **Contextual entities** — abstract things that live *only* in the
  metadata: the experiment, the equipment instances, the people involved.

We connect them with schema.org properties so a machine (or a human reader)
can navigate the graph: *"`temperature_T4750.csv` is **about** equipment
**T4750** which is **part of** the plant **C01 reference P&ID**, recorded
during **experiment E1** by **Jane Doe**."*


In [ ]:
from rocrate.model.contextentity import ContextEntity

crate = ROCrate()

# --- 5.1 Root metadata of the crate -----------------------------------------
crate.name        = "Demo RO-Crate: DEXPI plant setup with PAT data"
crate.description = (
    "Tutorial Research Object Crate bundling the DEXPI 1.3 reference plant, "
    "synthetic temperature data recorded on tank T4750, and a small "
    "temperature-profile analysis. Built with pyDEXPI + ro-crate-py."
)
crate.datePublished = datetime.now(timezone.utc)
crate.license       = "https://creativecommons.org/licenses/by/4.0/"
crate.keywords      = ["DEXPI", "P&ID", "FAIR", "process engineering",
                       "RO-Crate", "PAT data"]

# --- 5.2 Author (a contextual Person entity) --------------------------------
author = crate.add(ContextEntity(crate, "#author-jane", properties={
    "@type": "Person",
    "name": "Jane Doe",
    "affiliation": "Example University, Department of Chemical Engineering",
}))
crate.root_dataset["author"] = author

print("Root crate metadata initialised.")


✅ Root crate metadata initialised.


### 5.3 Register the **plant setup** (DEXPI file + equipment as research objects)

Each piece of equipment from the DEXPI plant becomes a **contextual entity**
of type `IndividualProduct`. The DEXPI file itself is registered as a
**file data entity** that we mark as the `mainEntity` describing the plant.


In [ ]:
# --- Folder describing the plant setup --------------------------------------
plant_setup_entity = crate.add_dataset(
    plant_setup_dir,
    dest_path="plant_setup",
    properties={
        "name": "Plant setup",
        "description": "DEXPI representation of the plant + its tagged equipment.",
    },
)

# --- The DEXPI Proteus XML --------------------------------------------------
dexpi_entity = crate.add_file(
    dexpi_in_crate,
    dest_path=f"plant_setup/{DEXPI_FILE.name}",
    properties={
        "name": "DEXPI 1.3 reference P&ID (Proteus XML)",
        "description": "Plant topology in DEXPI Proteus XML format.",
        "encodingFormat": "application/xml",
        "conformsTo": {"@id": "https://dexpi.org/"},
    },
)
crate.mainEntity = dexpi_entity  # the central piece described by the crate

# --- One contextual entity per tagged DEXPI equipment -----------------------
equipment_entities: dict[str, ContextEntity] = {}
for eq in equipment:
    ent = crate.add(ContextEntity(crate, f"#equipment-{eq['tag']}", properties={
        "@type": "IndividualProduct",
        "name": eq["tag"],
        "description": f"{eq['type']} extracted from the DEXPI plant via pyDEXPI.",
        "identifier": eq["uuid"],          # the DEXPI UUID — globally unique
        "additionalType": eq["type"],      # e.g. CentrifugalPump, Tank, ...
        "isPartOf": dexpi_entity,          # link back to the source DEXPI file
    }))
    equipment_entities[eq["tag"]] = ent

# Attach the equipment list to the plant_setup folder
plant_setup_entity["hasPart"] = [dexpi_entity, *equipment_entities.values()]

print(f"Registered {len(equipment_entities)} equipment items as research objects:")
for tag, ent in equipment_entities.items():
    print(f"   • {tag:6s}  →  {ent.id}")


✅ Registered 5 equipment items as research objects:
   • H1007   →  #equipment-H1007
   • H1008   →  #equipment-H1008
   • P4711   →  #equipment-P4711
   • P4712   →  #equipment-P4712
   • T4750   →  #equipment-T4750


### 5.4 Register the **data** (PAT measurements linked to equipment)

The CSV file gets a schema.org `about` link pointing to the **T4750**
equipment entity. We also describe a temperature **sensor** as a context
entity so future readers know *what* produced the measurements.


In [ ]:
# Folder for the experimental data
data_entity = crate.add_dataset(
    data_dir,
    dest_path="data",
    properties={
        "name": "Experimental PAT data",
        "description": "Process Analytical Technology measurements collected "
                       "during the experiment.",
    },
)

# A contextual entity describing the physical sensor used
temp_sensor = crate.add(ContextEntity(crate, "#sensor-T4750-temperature",
    properties={
        "@type": ["IndividualProduct", "SoftwareApplication"],
        "name": "Temperature sensor on T4750",
        "description": "PT-100 temperature sensor mounted on tank T4750.",
        "measurementTechnique": "PT-100 resistance thermometer",
        "unitText": "degree Celsius",
        "isPartOf": equipment_entities["T4750"],
    },
))

# The CSV file as a data entity, linked to both the equipment and the sensor
temperature_csv_entity = crate.add_file(
    temperature_csv,
    dest_path=f"data/{temperature_csv.name}",
    properties={
        "name": "Temperature time-series for equipment T4750",
        "description": "5-minute resolution temperature measurements (°C) "
                       "of the contents of tank T4750 during experiment E1.",
        "encodingFormat": "text/csv",
        "about": equipment_entities["T4750"],   # ← THE crucial link
        "wasGeneratedBy": temp_sensor,
        "variableMeasured": "temperature",
        "unitText": "degree Celsius",
    },
)

data_entity["hasPart"] = [temperature_csv_entity]

print(f"Linked '{temperature_csv.name}' to equipment "
      f"{equipment_entities['T4750']['name']} and sensor "
      f"{temp_sensor['name']}")


✅ Linked 'temperature_T4750.csv' to equipment T4750 and sensor Temperature sensor on T4750


### 5.5 Register the **analysis** (script + outputs + provenance)

The analysis is described as a `CreateAction` that links four things together:

| Property      | Meaning                              | Our value                       |
|---------------|--------------------------------------|---------------------------------|
| `agent`       | *Who* performed the action           | Jane Doe                        |
| `instrument`  | *With which tool/script*             | `temperature_profile.py`        |
| `object`      | *On which input data*                | `data/temperature_T4750.csv`    |
| `result`      | *Producing which outputs*            | `temperature_profile.png` + `.md` |

Crucially we also register the **analysis script itself** as a
`SoftwareSourceCode` entity inside the crate. This is what makes the
research object *re-executable*: a future reader can re-run the same
analysis on the same (or new!) data without leaving the crate.


In [37]:
analysis_dir_entity = crate.add_dataset(
    analysis_dir,
    dest_path="analysis",
    properties={
        "name": "Temperature profile analysis",
        "description": "Analysis script + descriptive statistics + plot of "
                       "the T4750 temperature time-series.",
    },
)

# The actual analysis script — registered as SoftwareSourceCode so a machine
# (or a human) immediately knows this file is *executable code*, not data.
analysis_script_entity = crate.add_file(
    analysis_script,
    dest_path=f"analysis/{analysis_script.name}",
    properties={
        "@type": ["File", "SoftwareSourceCode"],
        "name": "Temperature profile analysis script",
        "description": "Standalone Python script that reads the temperature "
                       "CSV and writes a PNG + Markdown report. Re-runnable "
                       "with: `python temperature_profile.py <csv> <out_dir>`.",
        "encodingFormat": "text/x-python",
        "programmingLanguage": "Python",
        "codeRepository": "https://github.com/process-intelligence-research/pyDEXPI",
        "runtimePlatform": "Python 3.12",
        "softwareRequirements": ["pandas", "matplotlib"],
    },
)

profile_png_entity = crate.add_file(
    profile_png,
    dest_path=f"analysis/{profile_png.name}",
    properties={
        "name": "Temperature profile plot",
        "encodingFormat": "image/png",
        "about": equipment_entities["T4750"],
    },
)
profile_md_entity = crate.add_file(
    report_md,
    dest_path=f"analysis/{report_md.name}",
    properties={
        "name": "Temperature profile analysis report",
        "encodingFormat": "text/markdown",
        "about": equipment_entities["T4750"],
    },
)

# The CreateAction now has a complete provenance chain:
#   agent  -- who did it
#   instrument -- with which tool/script
#   object -- on which input data
#   result -- producing which outputs
analysis_action = crate.add(ContextEntity(crate, "#action-temperature-analysis",
    properties={
        "@type": "CreateAction",
        "name": "Compute T4750 temperature profile",
        "description": "Pandas describe() + matplotlib line plot of the temperature time-series.",
        "agent": author,
        "instrument": analysis_script_entity,    # ← THE workflow / script
        "object": [temperature_csv_entity],
        "result": [profile_png_entity, profile_md_entity],
        "endTime": datetime.now(timezone.utc).isoformat(),
    },
))

analysis_dir_entity["hasPart"] = [
    analysis_script_entity, profile_png_entity, profile_md_entity,
]
print("✅ Analysis registered with full provenance: agent → instrument → object → result.")


✅ Analysis registered with full provenance: agent → instrument → object → result.


## 6 Write the crate

`crate.write()` materialises the metadata file (`ro-crate-metadata.json`)
inside the target folder. All the data files are already in place from
section 4, so this only generates / updates the index.


In [38]:
crate.write(CRATE_DIR)

print(f"📦 Crate written to: {CRATE_DIR.relative_to(PROJECT_ROOT)}\n")
print("Final crate contents:")
for p in sorted(CRATE_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(CRATE_DIR)}  ({p.stat().st_size/1024:.1f} KB)")


📦 Crate written to: ro_crate_output/plant_demo

Final crate contents:
  analysis/temperature_profile.md  (0.3 KB)
  analysis/temperature_profile.png  (62.4 KB)
  analysis/temperature_profile.py  (2.2 KB)
  data/temperature_T4750.csv  (6.3 KB)
  plant_setup/C01V04-VER.EX01.xml  (435.3 KB)
  ro-crate-metadata.json  (9.5 KB)


## 7 Inspect the generated `ro-crate-metadata.json`

This is the file that turns a regular folder into a Research Object Crate.
It is plain **JSON-LD**, using the [schema.org](https://schema.org/)
vocabulary, so any metadata harvester (Zenodo, B2SHARE, Dataverse, …) can
ingest it without further conversion.


In [39]:
import json

metadata_file = CRATE_DIR / "ro-crate-metadata.json"
metadata = json.loads(metadata_file.read_text())

print(f"@context: {metadata['@context']}")
print(f"\n# of entities described: {len(metadata['@graph'])}\n")

# Show a few interesting entities
def find(eid):
    return next((e for e in metadata["@graph"] if e["@id"] == eid), None)

for eid in ("./",
            "plant_setup/C01V04-VER.EX01.xml",
            "#equipment-T4750",
            "data/temperature_T4750.csv",
            "#action-temperature-analysis"):
    print("──", eid)
    print(json.dumps(find(eid), indent=2))
    print()


@context: https://w3id.org/ro/crate/1.2/context

# of entities described: 18

── ./
{
  "@id": "./",
  "@type": "Dataset",
  "author": {
    "@id": "#author-jane"
  },
  "datePublished": "2026-04-28T10:15:07.919060+00:00",
  "description": "Tutorial Research Object Crate bundling the DEXPI 1.3 reference plant, synthetic temperature data recorded on tank T4750, and a small temperature-profile analysis. Built with pyDEXPI + ro-crate-py.",
  "hasPart": [
    {
      "@id": "plant_setup/"
    },
    {
      "@id": "plant_setup/C01V04-VER.EX01.xml"
    },
    {
      "@id": "data/"
    },
    {
      "@id": "data/temperature_T4750.csv"
    },
    {
      "@id": "analysis/"
    },
    {
      "@id": "analysis/temperature_profile.py"
    },
    {
      "@id": "analysis/temperature_profile.png"
    },
    {
      "@id": "analysis/temperature_profile.md"
    }
  ],
  "keywords": [
    "DEXPI",
    "P&ID",
    "FAIR",
    "process engineering",
    "RO-Crate",
    "PAT data"
  ],
  "license": "h

---

## Recapitulation

You just built an **interoperable and reusable Research Object** for a 
process-engineering experiment. Notice how the metadata file makes every
relationship explicit and machine-actionable:

- The **plant** (`plant_setup/C01V04-VER.EX01.xml`) is the *main entity*.
- Each **equipment** is an `IndividualProduct` carrying its DEXPI UUID.
- The **temperature CSV** is `about` equipment `T4750` and was
  `wasGeneratedBy` a documented sensor.
- The **analysis** is a `CreateAction` whose `object` is the CSV and
  whose `result` is the plot + report — full provenance.

How does this address each FAIR principle so far?

| Principle          | What we did |
|--------------------|--------------------------------------------------------------------------------|
| **F**indable       | |
| **A**ccessible     | Provenance via `wasGeneratedBy` and `CreateAction`; clear license at the root  |
| **I**nteroperable  | Vocabularies: DEXPI 1.3 + schema.org; no proprietary formats                   |
| **R**eusable       | Provenance via `wasGeneratedBy` and `CreateAction`; clear license at the root  |

What about making the data now Findable?

### Next Steps

1. Zip the crate with `crate.write_zip("plant_demo.zip")` and upload it to
   [Zenodo](https://zenodo.org/) - you'll get a citable DOI (ke )
2. Validate the crate with the official
   [`rocrate-validator`](https://github.com/ResearchObject/ro-crate-validator-py).

### Further Reading

- RO-Crate specification: <https://www.researchobject.org/ro-crate/>
- `ro-crate-py` documentation: <https://github.com/ResearchObject/ro-crate-py>
- DEXPI standard: <https://dexpi.org/>
- pyDEXPI: <https://github.com/process-intelligence-research/pyDEXPI>
- FAIR principles: <https://www.go-fair.org/fair-principles/>
